## Trainning The Evaluation Function Weights


In [12]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import StandardScaler

### Representing Data in 2D grid format

In [13]:

EMPTY = 0
# Piece Representations
BLACK_PAWN = -1
BLACK_ROOK = -2
BLACK_KNIGHT = -3
BLACK_BISHOP = -4
BLACK_QUEEN = -5
BLACK_KING = -6

WHITE_PAWN = 1
WHITE_ROOK = 2
WHITE_KNIGHT = 3
WHITE_BISHOP = 4
WHITE_QUEEN = 5
WHITE_KING = 6


def fromFEN(fen):
    # Creates an Empty Board of 8x8
    board = []
    for i in range(8):
        row=[]
        for j in range(8):
            row.append(EMPTY)
        board.append(row)

    row = 0
    col = 0

    for c in fen:
        # if c reaches an empty character then board representation ends
        if c==' ':
            break
        # if / is encountered move to next row
        if c == '/':
            row += 1
            col = 0
        # if a digit is encountered then skip that many consecutive squares
        elif c.isdigit():
            col += int(c)
        # if character is found then place it on current row and column
        else:
            if c == 'P':
                board[row][col] = WHITE_PAWN
            elif c == 'R':
                board[row][col] = WHITE_ROOK
            elif c == 'N':
                board[row][col] = WHITE_KNIGHT
            elif c == 'B':
                board[row][col] = WHITE_BISHOP
            elif c == 'Q':
                board[row][col] = WHITE_QUEEN
            elif c == 'K':
                board[row][col] = WHITE_KING

            elif c == 'p':
                board[row][col] = BLACK_PAWN
            elif c == 'r':
                board[row][col] = BLACK_ROOK
            elif c == 'n':
                board[row][col] = BLACK_KNIGHT
            elif c == 'b':
                board[row][col] = BLACK_BISHOP
            elif c == 'q':
                board[row][col] = BLACK_QUEEN
            elif c == 'k':
                board[row][col] = BLACK_KING

            col += 1

    return board


data=pd.read_csv("data_small_train.csv")
data["fen"][0]
tp=fromFEN(data["fen"][0])
tp

[[0, 0, 0, 0, 0, 0, -6, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0, 6, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0]]

### getting features from current board representation

In [14]:
import subprocess

X=[]

def getFeatures(dataFrame):
    boards = []

    for data_str in dataFrame["fen"]:
        arr = fromFEN(data_str)

        for row in arr:
            boards.append(" ".join(map(str, row)))

    inp = f"{len(dataFrame)}\n" + "\n".join(boards) + "\n"

    result = subprocess.run(
        ["../feature"],
        input=inp,
        text=True,
        capture_output=True
    )

    return [
        list(map(float, line.split()))
        for line in result.stdout.strip().splitlines()
    ]

X=getFeatures(data)

X_train=pd.DataFrame(
    X,
    columns=["material","mobility"]
)
X_train.head()

,material,mobility
0,95.0,13.0
1,50.0,3.0
2,75.0,14.0
3,-243.0,-93.0
4,208.0,35.0


In [17]:
scale=StandardScaler()
scale.fit(X_train)
X_train_scaled=scale.transform(X_train)
Y_train=data[["score"]]
model=LinearRegression()
model.fit(X_train_scaled,Y_train)
Y_pred=model.predict(X_train_scaled)
err = root_mean_squared_error(Y_train,Y_pred)
print("Error in train set: ",err)

Error in train set:  649.9370817407994
